# Curriculum 05 · Lab 2 — Multi-query: one question in, several search queries out

**Goal:** Remove phrasing luck from retrieval. A single question is a single
point in embedding space, but the answer may live near several *different*
points — the same fact phrased as "capital of Uruguay", "Montevideo",
"Uruguay's largest city" each lands in a different region of the store.
Top-k samples ONE region; multi-query samples many.

```
Retriever : MultiQueryRetriever (langchain-classic) — from_llm(...)
Generator : llama-3.3-70b-versatile (ChatGroq) — 3+ query variants per question
Retriever : store.as_retriever(k=3) — LangChain BaseRetriever contract
Merge     : unique union (include_original=True — never worse than top-k)
Embedding : BGE (BAAI/bge-base-en-v1.5, local, CPU)
Data      : rag-mini-wikipedia — first 100 passages, questions 1606/1610/1626
```

**Why multi-query:** the LLM generates 3+ phrasings of the question, every
phrasing is retrieved, and the results are merged into one deduplicated
union — so a fact reachable through ANY phrasing surfaces. Unlike HyDE
(lab 4) the variants stay *questions*; unlike step-back (lab 5) they stay
*specific*. `include_original=True` guarantees the union always contains the
plain top-k result.

This is the second lab of track 05-query-transformation (see
`.omo/plans/layer1-rag-playbook.md`).


## 0 · Setup — environment, imports & repo paths

**WHAT:** Installs the lab's dependencies (a no-op if already present),
loads `GROQ_API_KEY` from the repo-root `.env`, and puts the repo root on
`sys.path` so every repo-relative path behaves exactly like the lab script.

**WHY:** Everything embeds **locally** with BGE. The only API call is the
query *generator* — Groq's `llama-3.3-70b-versatile` (a commented Gemini
alternative is kept in the source). This lab uses LangChain-native FAISS
(`langchain-community`) because `MultiQueryRetriever` wraps a LangChain
`BaseRetriever` — the same shape as track 04 lab 5.

**WHAT TO EXPECT:** no output from the pip cell (packages already
installed), a silent import from the second (a `DeprecationWarning` about
`langchain-community` is expected and harmless). The BGE model loads lazily
when the experiment cell first calls it; the Groq key is read from `.env`.


In [1]:
# Lab-specific dependencies (already in requirements.txt — the install is a
# no-op safety net for fresh environments):
#   sentence-transformers -> local BGE embeddings (langchain_huggingface)
#   faiss-cpu             -> the FAISS vector store (langchain_community)
#   langchain-classic     -> MultiQueryRetriever
#   langchain-groq        -> ChatGroq (the query generator LLM)
#   python-dotenv         -> loads GROQ_API_KEY from the repo-root .env
#   pandas                -> reads the passages/test.parquet corpus
%pip install sentence-transformers faiss-cpu langchain-classic langchain-groq python-dotenv pandas



[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the kernel's
# working directory — this works whether the kernel launches from the repo
# root (like the lab script) or from the notebook's own folder (Jupyter's
# default) — then cd into it so every repo-relative path behaves exactly
# like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

load_dotenv(REPO_ROOT / ".env")  # GROQ_API_KEY lives in the repo-root .env

from langchain_classic.retrievers.multi_query import (  # noqa: E402
    MultiQueryRetriever,
)
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_groq import ChatGroq  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402
# (Gemini alternative: from langchain_google_genai import ChatGoogleGenerativeAI)


/tmp/ipykernel_1424901/3635989388.py:30: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS  # noqa: E402


## 1 · Configuration — the experiment's knobs

**WHAT:** The corpus constants (`N_PASSAGES = 100`, `QUESTION_IDS =
[1606, 1610, 1626]` — the same three questions as lab 01, for direct
comparison) plus `TOP_K = 3` (per-variant retrieval depth — the union is
bigger than k) and `LLM_MODEL = "llama-3.3-70b-versatile"`.

**WHY:** `include_original=True` is the load-bearing choice: the user's own
query joins the LLM's variants, so the union can never be *worse* than plain
top-k — the gate exploits exactly that guarantee.


In [3]:
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus (keeps runtime low)
QUESTION_IDS = [1606, 1610, 1626]  # same questions as lab 01, for comparison
TOP_K = 3  # per-variant retrieval depth; the union is bigger than k
LLM_MODEL = "llama-3.3-70b-versatile"  # Groq is the query *generator*, never the embedder
# (Gemini alternative: LLM_MODEL = "gemini-2.5-flash" — needs GOOGLE_API_KEY in .env)
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
PREVIEW = 62  # max characters of passage text shown next to each hit


## 2 · Load — corpus + questions from the fresh parquet files

**WHAT:** `load_passages` pulls the first `n` passages (text + ids) from
`passages.parquet`; `load_questions` pulls specific rows by id from
`test.parquet`; `preview` flattens a passage for one-line printing.

**WHY:** Identical helpers to lab 01 — the corpus and questions are the
same, so every difference in the output is caused by the query
transformation, not the data.


In [4]:
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


## 3 · Experiment — LLM variants -> per-variant retrieval -> union

**WHAT:** `run_experiment` embeds the 100-passage subset once, builds the
LangChain FAISS store and a `BaseRetriever` from it, wraps it in
`MultiQueryRetriever.from_llm(..., include_original=True)`, then per question
generates the variants (one Groq call, via the wrapper's `llm_chain`) and
runs the full multi-query retrieval — recording the variants, the deduplicated
union, and the timings.

**WHY:** The variant list and the union must come from the same pipeline so
the demo can show exactly what the LLM wrote and exactly what the retriever
merged — one shared `exp`.


In [5]:
def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)

    # --- Embed locally (BGE) and index in-memory with langchain-native FAISS -
    chunks = [
        Document(page_content=t, metadata={"id": pid})
        for t, pid in zip(passage_texts, passage_ids)
    ]
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME, encode_kwargs={"normalize_embeddings": True}
    )
    t0 = time.perf_counter()
    store = FAISS.from_documents(chunks, embedder)
    index_s = time.perf_counter() - t0

    # --- Base retriever (LangChain contract) + the multi-query wrapper --------
    base_retriever = store.as_retriever(search_kwargs={"k": TOP_K})
    # include_original=True: the user's own query joins the LLM's variants, so
    # the union can never be WORSE than plain top-k.
    llm = ChatGroq(model=LLM_MODEL, temperature=0.0)
    multi_retriever = MultiQueryRetriever.from_llm(
        retriever=base_retriever, llm=llm, include_original=True
    )

    # --- Per question: generated variants + the merged union ------------------
    results = []
    for qid, qtext in questions:
        t0 = time.perf_counter()
        variants = multi_retriever.llm_chain.invoke({"question": qtext})
        gen_s = time.perf_counter() - t0
        t0 = time.perf_counter()
        union = multi_retriever.invoke(qtext)
        union_s = time.perf_counter() - t0
        results.append(
            {
                "qid": qid,
                "question": qtext,
                "variants": variants,
                "gen_s": gen_s,
                "union": union,
                "union_s": union_s,
            }
        )

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "indexed": len(passage_texts),
        "index_s": index_s,
        "results": results,
    }


## 4 · Run — execute the experiment

**WHAT:** Calls `run_experiment()` — embedding, indexing, and one Groq
generation call per question (3 variants each) take a few seconds. The
artifact dict is kept as `exp`.

**WHY:** As in lab 01, demo and gate both read this single `exp`. The gate's
union checks depend on the actual generated variants, so the LLM calls must
happen here, once.


In [6]:
exp = run_experiment()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 5 · Demo — read the artifact

**WHAT:** `print_demo` prints the corpus summary, then per question the
LLM-generated variants, the union size vs `k`, and the top-k of the union
with passage previews.

**WHY:** The union size is the headline — expect more than `k=3` unique docs
once different phrasings retrieve different passages. The variant list shows
the generator doing the work: 3+ different ways to ask the same thing, each
probing a different region of the store.


In [7]:
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 02 — Multi-query: one question in, several search queries out")
    print(f"{BGE_MODEL_NAME} (local) -> FAISS top-{TOP_K} -> {LLM_MODEL} generator")
    print("=" * 66)

    print(f"\n[1] Corpus (deterministic subset, no randomness):")
    print(f"    {exp['indexed']} passages (first {N_PASSAGES} of 3200, ids {exp['passage_ids'][0]}..{exp['passage_ids'][-1]})")
    print(f"    FAISS index built in {exp['index_s']:.3f}s over local BGE embeddings")

    print(f"\n[2] Generated variants -> union (per question):")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"]}] "{r["question"]}"')
        print(f"      variants ({len(r['variants'])}, {r['gen_s']:.1f}s):")
        for v in r["variants"]:
            print(f"        - {v}")
        print(f"      union: {len(r['union'])} unique docs "
              f"(k={TOP_K} per variant, {r['union_s']:.1f}s)")
        for rank, doc in enumerate(r["union"][:TOP_K], 1):
            pid = doc.metadata.get("id", "?")
            print(f"        {rank}. [passage {pid}] {preview(doc.page_content)}")
        if len(r["union"]) > TOP_K:
            print(f"        … {len(r['union']) - TOP_K} more unique docs beyond top-{TOP_K}")

    print("\n[3] Takeaway")
    print("    Multi-query trades one cheap LLM call per question for several")
    print("    retrieval passes over different phrasings, then merges the")
    print("    unique results. The union is never worse than plain top-k")
    print("    (the original query is included), and a fact reachable only")
    print("    through a different phrasing finally has a chance to surface.")


In [8]:
print_demo(exp)


Lab 02 — Multi-query: one question in, several search queries out
BAAI/bge-base-en-v1.5 (local) -> FAISS top-3 -> llama-3.3-70b-versatile generator

[1] Corpus (deterministic subset, no randomness):
    100 passages (first 100 of 3200, ids 0..99)
    FAISS index built in 1.111s over local BGE embeddings

[2] Generated variants -> union (per question):

    Q[1606] "Is Uruguay's capital Montevideo?"
      variants (5, 0.4s):
        - What is the capital city of Uruguay?
        -  
        - Is Montevideo the main city in Uruguay?
        -  
        - What city serves as the capital of Uruguay, and is it Montevideo?
      union: 8 unique docs (k=3 per variant, 0.4s)
        1. [passage 36] Montevideo, Uruguay's capital.
        2. [passage 0] Uruguay (official full name in  ; pron.  , Eastern Republic of...
        3. [passage 15] Uruguay's capital, Montevideo, was founded by the Spanish in t...
        … 5 more unique docs beyond top-3

    Q[1610] "Who founded Montevideo?"
      var

## 6 · Verification gate — the same checks the .py runs

**WHAT:** Runs the exact `verify_gate`: exactly `N_PASSAGES` indexed, per
question — at least one generated variant, the union deduplicated, union size
>= TOP_K (the `include_original` guarantee), and the content check that the
union carries the answer's keyword (montevideo / spanish / 1930).

**WHY:** `python 02-multi-query.py --verify` must print 13/13 PASS; this cell
proves the notebook reproduces the verified `.py` exactly. The checks are
pinned to retrieval outcomes, not exact LLM wording, so they stay stable
across runs.


In [9]:
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # Structural properties (no LLM involved).
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))

    # Multi-query properties, pinned to what survives LLM wording variance.
    for r in exp["results"]:
        tag = f"Q{r['qid']}"

        # The generator must return at least one variant.
        checks.append((f"{tag} generated >=1 query variant", len(r["variants"]) >= 1))

        # The union is deduplicated: unique page content, no repeats.
        contents = [d.page_content for d in r["union"]]
        checks.append((f"{tag} union is deduplicated",
                       len(contents) == len(set(contents))))

        # include_original=True guarantees the union covers plain top-k, so it
        # is always >= TOP_K distinct documents.
        checks.append((f"{tag} union size >= TOP_K", len(r["union"]) >= TOP_K))

        # Content check: the union must carry the answer's keyword.
        joined = " ".join(d.page_content for d in r["union"]).lower()
        if r["qid"] == 1606:
            kw = "montevideo"
        elif r["qid"] == 1610:
            kw = "spanish"
        else:  # 1626
            kw = "1930"
        checks.append((f"{tag} union retains '{kw}'", kw in joined))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


In [10]:
verify_gate(exp)


verification gate:
  [PASS] exactly 100 passages indexed
  [PASS] Q1606 generated >=1 query variant
  [PASS] Q1606 union is deduplicated
  [PASS] Q1606 union size >= TOP_K
  [PASS] Q1606 union retains 'montevideo'
  [PASS] Q1610 generated >=1 query variant
  [PASS] Q1610 union is deduplicated
  [PASS] Q1610 union size >= TOP_K
  [PASS] Q1610 union retains 'spanish'
  [PASS] Q1626 generated >=1 query variant
  [PASS] Q1626 union is deduplicated
  [PASS] Q1626 union size >= TOP_K
  [PASS] Q1626 union retains '1930'


0